# Fake News Detection — Colab Eğitim Rehberi

Bu notebook kodu GitHub'dan, ham WELFake verisini Drive'dan alır. Eğitim Colab'ın hızlı `/content` diskinde çalışır. Checkpoint ve bitmiş sonuçlar Drive'da kalır.

İlk çalıştırmadan önce aşağıdaki `GITHUB_REPO_URL` değerini kendi GitHub depo adresinizle değiştirin.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PROJECT = Path('/content/drive/MyDrive/fake_news_detection')
RAW_DATASET = DRIVE_PROJECT / 'data' / 'raw' / 'WELFake_Dataset.csv'
# Temizleme kuralları değişirse bu klasör adını v2 yaparak yeni sürüm oluşturun.
PREPARED_DATA_ON_DRIVE = DRIVE_PROJECT / 'data' / 'prepared' / 'welfake_temiz_v1_seed42'
DRIVE_ARTIFACTS = DRIVE_PROJECT / 'artifacts'
CHECKPOINT_ROOT = DRIVE_ARTIFACTS / 'checkpointler'

if not RAW_DATASET.is_file():
    raise FileNotFoundError(f'Veri seti bulunamadı: {RAW_DATASET}')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
PREPARED_DATA_ON_DRIVE.parent.mkdir(parents=True, exist_ok=True)
print('Drive hazır.')

In [ ]:
# Kaynak kod bu depodan alınır.
GITHUB_REPO_URL = 'https://github.com/burak-dursunn/fake-news-detection.git'
PROJECT_DIR = Path('/content/fake-news-detection')

if not PROJECT_DIR.exists():
    !git clone {GITHUB_REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd /content/fake-news-detection
!pip install -q -r requirements.txt
!nvidia-smi

## 1. Ham WELFake verisini Colab'a kopyalama ve temizleme

Bu adım Drive'daki **ham** CSV dosyasını değiştirmez. İlk çalıştırmada temiz veriyi üretir ve Drive'a kaydeder. Sonraki model eğitimlerinde aynı temiz veri ve aynı sabit eğitim/doğrulama/test bölmesi Drive'dan tekrar kullanılır.

In [ ]:
# Ham veri Drive'da kalır; eğitim için gereken kopyalar Colab'ın hızlı diskine alınır.
import shutil
import json

LOCAL_DATASET = Path('/content/WELFake_Dataset.csv')
LOCAL_PREPARED_DATA = Path('/content/prepared_data')
CLEAN_DATA_FILE = 'welfake_temiz.csv'
SPLIT_FILE = 'welfake_split_seed42.csv'

# Önceden hazırlanmış veri varsa yeniden temizleme yapılmaz.
prepared_exists = (PREPARED_DATA_ON_DRIVE / CLEAN_DATA_FILE).is_file() and (PREPARED_DATA_ON_DRIVE / SPLIT_FILE).is_file()
if prepared_exists:
    print('Drive üzerinde daha önce hazırlanmış temiz veri bulundu; yeniden kullanılıyor.')
    shutil.copytree(PREPARED_DATA_ON_DRIVE, LOCAL_PREPARED_DATA, dirs_exist_ok=True)
else:
    print('Temiz veri bulunamadı; ham WELFake temizleniyor.')
    shutil.copy2(RAW_DATASET, LOCAL_DATASET)
    !python -m src.prepare_data --input {LOCAL_DATASET} --output-dir {LOCAL_PREPARED_DATA}
    shutil.copytree(LOCAL_PREPARED_DATA, PREPARED_DATA_ON_DRIVE, dirs_exist_ok=True)
    print(f"Temiz veri Drive'a kaydedildi: {PREPARED_DATA_ON_DRIVE}")

cleaning_report = json.loads((LOCAL_PREPARED_DATA / 'welfake_temizleme_raporu.json').read_text())
split_report = json.loads((LOCAL_PREPARED_DATA / 'welfake_split_ozeti.json').read_text())
print('\nTEMİZLEME RAPORU')
for key, value in cleaning_report.items():
    print(f'- {key}: {value}')
print('\nVERİ BÖLMESİ')
print(split_report['split_counts'])
print('\nTemiz veri ve sabit train/validation/test bölmesi hazır.')

In [ ]:
# İlk deney: DistilBERT, 256 token.
# Diğer deneyler için yalnızca config dosyasını değiştirin.
CONFIG_PATH = Path('configs/01_DistilBERT_256Token.json')
EXPERIMENT_NAME = '01_DistilBERT_256Token'
LOCAL_OUTPUT = Path('/content/experiments')

!python -m src.run_experiment \
  --config {CONFIG_PATH} \
  --data-dir {LOCAL_PREPARED_DATA} \
  --output-root {LOCAL_OUTPUT} \
  --checkpoint-root {CHECKPOINT_ROOT}


In [ ]:
# Eğitim bittiğinde tüm sunum grafikleri, metrikler ve en iyi model Drive'a kopyalanır.
SOURCE = LOCAL_OUTPUT / EXPERIMENT_NAME
DESTINATION = DRIVE_ARTIFACTS / 'deneyler' / EXPERIMENT_NAME
DESTINATION.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(SOURCE, DESTINATION, dirs_exist_ok=True)
print(f'Sonuçlar kaydedildi: {DESTINATION}')

In [ ]:
# Sunumda kullanılacak grafiklere Colab içinde hızlı bakış.
from IPython.display import Image, display

for file_name in [
    '01_Egitim_Dogrulama_Kaybi.png',
    '02_Dogruluk_ve_F1_Egrisi.png',
    '03_ROC_Egrisi.png',
    '04_Precision_Recall_Egrisi.png',
    '05_Karisiklik_Matrisi.png',
]:
    print(file_name)
    display(Image(filename=str(SOURCE / file_name)))